# Feature Engineering

In [1]:
# Importing libraries

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

## Loading the Data

In [2]:
# Loading the data

df = pd.read_parquet('../data/df_clean.parquet')
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df['isFraud'].mean().round(4)}')

Shape: (590540, 423)
Fraud rate: 0.035


## Step 1 | Separating the Target from the Features

In [3]:
# Separating the target variable from the features

X = df.drop(columns=['isFraud', 'TransactionID'])
y = df['isFraud']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Fraud cases: {y.sum()} ({y.mean()*100:.1f}%)')

Features shape: (590540, 421)
Target shape: (590540,)
Fraud cases: 20663 (3.5%)


## Step 2 | Removing Features with Near-Zero Variance

In [4]:
# Removing features with near-zero variance

selector = VarianceThreshold(threshold=0.01)
selector.fit(X)

kept_cols = X.columns[selector.get_support()].tolist()
dropped_cols = X.columns[~selector.get_support()].tolist()

print(f'Columns before: {X.shape[1]}')
print(f'Columns dropped (near-zero variance): {len(dropped_cols)}')
print(f'Columns kept: {len(kept_cols)}')
print(f'\nDropped: {dropped_cols}')

X = X[kept_cols]

Columns before: 421
Columns dropped (near-zero variance): 0
Columns kept: 421

Dropped: []


## Step 3 | Removing Features Highly-Correlated with Each Other

In [5]:
# Removing features highly-correlated with each other

# Computing the correlation matrix
corr_matrix = X.corr().abs()

# Avoiding duplicate pairs (upper triangle only)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Finding columns with a correlation of above 0.95
dropped_cols_corr = [col for col in upper.columns if any(upper[col] > 0.95)]

print(f'Highly-correlated columns to drop (correlation > 0.95): {len(dropped_cols_corr)}')
print(f'Remaining columns: {X.shape[1] - len(dropped_cols_corr)}')

X = X.drop(columns=dropped_cols_corr)
print(f'Shape after correlation filter: {X.shape}')

Highly-correlated columns to drop (correlation > 0.95): 317
Remaining columns: 104
Shape after correlation filter: (590540, 104)


## Step 4 | Engineering New Features

In [6]:
# Engineering new features

# Transaction hour of the day
X['hour'] = (df['TransactionDT'] / 3600 % 24).astype(int)

# Transaction day of the week
X['day_of_week'] = (df['TransactionDT'] / (3600 * 24) % 7).astype(int)

# Indicator for transactions outside business hours (typically 10pm - 6am)
X['is_nighttime'] = ((X['hour'] >= 22) | (X['hour'] <= 6)).astype(int)

# Indicator for high-value transactions (typically above the 75th percentile)
amt_75 = X['TransactionAmt'].quantile(0.75)
X['is_high_value'] = (X['TransactionAmt'] > amt_75).astype(int)

# Indicator for whether purchaser email matches recipient email
X['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)

print('New features engineered:')
print(' hour - Transaction hour of the day')
print(' day_of_week - Transaction day of the week (0 = Monday, 6 = Sunday)')
print(' is_nighttime - Indicator for transactions outside business hours (typically 10pm - 6am)')
print(' is_high_value - Indicator for high-value transactions (typically above the 75th percentile)')
print(' email_match - Indicator for whether purchaser email matches recipient email')
print(f'\nFinal feature shape: {X.shape}')

New features engineered:
 hour - Transaction hour of the day
 day_of_week - Transaction day of the week (0 = Monday, 6 = Sunday)
 is_nighttime - Indicator for transactions outside business hours (typically 10pm - 6am)
 is_high_value - Indicator for high-value transactions (typically above the 75th percentile)
 email_match - Indicator for whether purchaser email matches recipient email

Final feature shape: (590540, 109)


## Sanity Checks for Newly-Engineered Features

In [7]:
# Sanity checks for newly-engineered features

print('Hour distribution (sample):')
print(X['hour'].value_counts().sort_index().head(5))

print('\nNighttime transactions:')
print(X['is_nighttime'].value_counts())

print('\nHigh-value transactions:')
print(X['is_high_value'].value_counts())

print('\nTransactions with matching emails:')
print(X['email_match'].value_counts())

print('\nFraud rate in nighttime transactions:')
print(df.loc[X.index].groupby(X['is_nighttime'])['isFraud'].mean().round(4))

print('\nFraud rate in high-value transactions:')
print(df.loc[X.index].groupby(X['is_high_value'])['isFraud'].mean().round(4))

print('\nFraud rate in transactions with matching emails:')
print(df.loc[X.index].groupby(X['email_match'])['isFraud'].mean().round(4))

Hour distribution (sample):
hour
0    37795
1    32797
2    26732
3    20802
4    14839
Name: count, dtype: int64

Nighttime transactions:
is_nighttime
0    360779
1    229761
Name: count, dtype: int64

High-value transactions:
is_high_value
0    444712
1    145828
Name: count, dtype: int64

Transactions with matching emails:
email_match
0    494039
1     96501
Name: count, dtype: int64

Fraud rate in nighttime transactions:
is_nighttime
0    0.0330
1    0.0381
Name: isFraud, dtype: float64

Fraud rate in high-value transactions:
is_high_value
0    0.0319
1    0.0444
Name: isFraud, dtype: float64

Fraud rate in transactions with matching emails:
email_match
0    0.0225
1    0.0988
Name: isFraud, dtype: float64


### Feature Engineering Insights

Newly-engineered features validated against fraud rate:
- is_nighttime: 3.3% vs 3.8% (weak signal)
- is_high_value: 3.2% vs 4.4% (moderate signal)
- email_match: 2.3% vs 9.9% (strong signal)

All three features will be retained for training

## Step 5 | Train/Test Split

In [8]:
# Train/Test Split

# Stratified to preserve fraud rate in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train fraud rate: {y_train.mean().round(4)}')
print(f'y_test fraud rate: {y_train.mean().round(4)}')

X_train shape: (472432, 109)
X_test shape: (118108, 109)
y_train fraud rate: 0.035
y_test fraud rate: 0.035


## Step 6 | Applying SMOTE to Train Data to Handle Class Imbalance

In [9]:
# Applying SMOTE to train data to handle class imbalance

print(f'Fraud cases in train data before SMOTE: {y_train.sum()} ({y_train.mean()*100:.1f}%)')

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f'Fraud cases in train data after SMOTE: {y_train_resampled.sum()} ({y_train_resampled.mean()*100:.1f}%)')
print(f'Train set size after SMOTE: {X_train_resampled.shape}')

Fraud cases in train data before SMOTE: 16530 (3.5%)
Fraud cases in train data after SMOTE: 455902 (50.0%)
Train set size after SMOTE: (911804, 109)


## Step 7 | Saving Feature Sets

In [10]:
# Saving feature sets

# Creating the folder to store features
os.makedirs('../data/features', exist_ok=True)

# Saving the resampled train data
pd.DataFrame(X_train_resampled, columns=X.columns).to_parquet('../data/features/X_train.parquet', index=False)
pd.DataFrame(y_train_resampled, columns=['isFraud']).to_parquet('../data/features/y_train.parquet', index=False)

# Saving test data
pd.DataFrame(X_test, columns=X.columns).to_parquet('../data/features/X_test.parquet', index=False)
pd.DataFrame(y_test, columns=['isFraud']).to_parquet('../data/features/y_test.parquet', index=False)

# Saving feature names
feature_names = X.columns.tolist()
pd.Series(feature_names).to_json('../data/features/feature_names.json')

print(f'X_train saved: {X_train_resampled.shape}')
print(f'y_train saved: {y_train_resampled.shape}')
print(f'X_test saved: {X_test.shape}')
print(f'y_test saved: {y_test.shape}')
print(f'Feature names saved: {len(feature_names)} features')

X_train saved: (911804, 109)
y_train saved: (911804,)
X_test saved: (118108, 109)
y_test saved: (118108,)
Feature names saved: 109 features


This concludes the EDA phase, where the dataset has been optimised and enriched with useful features, preparing it for the next phase which is model training.